# Pipeline configuration
Central configuration for the Adventure Works medallion pipeline. Every pipeline notebook loads it with `%run`.

**Widgets** (job parameters with the same name override the defaults):
* `catalog` – Unity Catalog catalog holding the `bronze`, `silver` and `gold` schemas
* `storage_account` – ADLS Gen2 account with the `raw`, `bronze`, `silver` and `gold` containers
* `datasets` – comma-separated dataset names to process, or `all`

In [0]:
# Defaults for interactive runs; job parameters with the same names override them
dbutils.widgets.text("catalog", "adventure_works_catalog", "Catalog")
dbutils.widgets.text("storage_account", "adventurework", "Storage account")
dbutils.widgets.text("datasets", "all", "Datasets (comma-separated or 'all')")

CATALOG = dbutils.widgets.get("catalog").strip()
STORAGE_ACCOUNT = dbutils.widgets.get("storage_account").strip()

In [0]:
def container_path(container: str) -> str:
    return f"abfss://{container}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"

RAW_PATH    = container_path("raw")
BRONZE_PATH = container_path("bronze")
SILVER_PATH = container_path("silver")
GOLD_PATH   = container_path("gold")

BRONZE_SCHEMA = f"{CATALOG}.bronze"
SILVER_SCHEMA = f"{CATALOG}.silver"
GOLD_SCHEMA   = f"{CATALOG}.gold"

STORAGE_CREDENTIAL = "adw_credentials"
EXTERNAL_LOCATIONS = {
    "adw_raw_location":    RAW_PATH,
    "adw_bronze_location": BRONZE_PATH,
    "adw_silver_location": SILVER_PATH,
    "adw_gold_location":   GOLD_PATH,
}

# Raw files are tab-separated text with a header row
RAW_CSV_OPTIONS = {"header": "true", "sep": "\t"}

In [0]:
# table name -> source file, storage folder and primary key (used for dedup and DQ checks)
DATASETS = {
    "product":            {"source_file": "Product.csv",           "folder": "Product",           "primary_key": ["ProductKey"]},
    "region":             {"source_file": "Region.csv",            "folder": "Region",            "primary_key": ["SalesTerritoryKey"]},
    "reseller":           {"source_file": "Reseller.csv",          "folder": "Reseller",          "primary_key": ["ResellerKey"]},
    "sales":              {"source_file": "Sales.csv",             "folder": "Sales",             "primary_key": ["SalesOrderNumber", "ProductKey"]},
    "salesperson":        {"source_file": "Salesperson.csv",       "folder": "Salesperson",       "primary_key": ["EmployeeKey"]},
    "salesperson_region": {"source_file": "SalespersonRegion.csv", "folder": "SalespersonRegion", "primary_key": ["EmployeeKey", "SalesTerritoryKey"]},
    "targets":            {"source_file": "Targets.csv",           "folder": "Targets",           "primary_key": ["EmployeeID", "TargetMonth"]},
}

_selected = dbutils.widgets.get("datasets").strip().lower()
SELECTED_DATASETS = list(DATASETS) if _selected in ("", "all") else [d.strip() for d in _selected.split(",") if d.strip()]
_unknown = set(SELECTED_DATASETS) - set(DATASETS)
if _unknown:
    raise ValueError(f"Unknown dataset(s) in 'datasets' widget: {sorted(_unknown)}. Valid: {list(DATASETS)}")

print(f"catalog={CATALOG} | storage_account={STORAGE_ACCOUNT} | datasets={SELECTED_DATASETS}")